In [31]:
import re
import requests
import base64
import json
from datetime import datetime
import pytz
import yaml
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

In [32]:
# ====== VirusTotal API Setup ======
API_KEY = '3adc7c41151f7947e3d013480f8d7cabe279e24eeb8ae84368664198fe17dbc7'  
BASE_URL = 'https://www.virustotal.com/api/v3'

def get_url_id(url):
    url_id = base64.urlsafe_b64encode(url.encode()).decode().strip("=")
    return url_id

def query_virustotal(ioc_dict):
    headers = {
        'x-apikey': API_KEY
    }

    ioc_value = ioc_dict['ioc']
    ioc_type = ioc_dict['type']

    if ioc_type == 'ip':
        url = f'{BASE_URL}/ip_addresses/{ioc_value}'
    elif ioc_type == 'domain':
        url = f'{BASE_URL}/domains/{ioc_value}'
    elif ioc_type == 'file':
        url = f'{BASE_URL}/files/{ioc_value}'
    else:
        print('Unsupported IOC type.')
        return None

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        parsed_result = parse_response(data)
        return parsed_result
    else:
        print(f'Error: {response.status_code}')
        return None


In [33]:
def parse_response(data):
    try:
        stats = data['data']['attributes']['last_analysis_stats']
        last_analysis_date = data['data']['attributes'].get('last_analysis_date', None)

        harmless = stats.get('harmless', 0)
        malicious = stats.get('malicious', 0)
        suspicious = stats.get('suspicious', 0)

        if last_analysis_date:
            readable_date = datetime.utcfromtimestamp(last_analysis_date).strftime('%Y-%m-%d %H:%M:%S')
        else:
            readable_date = "N/A"

        output = {
            "harmless": harmless,
            "malicious": malicious,
            "suspicious": suspicious,
            "last_analysis_date": readable_date
        }

        return output

    except KeyError as e:
        print(f'Missing field: {e}')
        return {}

In [34]:
# ====== Check IOC ======
def checkioc(ioc):
    ip_pattern = re.compile(r"^(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)$")
    domain_pattern = re.compile(r"^[a-zA-Z0-9][a-zA-Z0-9-]{1,61}[a-zA-Z0-9]\.[a-zA-Z]{2,}$")
    hash_pattern_md5 = re.compile(r"^[a-fA-F0-9]{32}$")
    hash_pattern_sha1 = re.compile(r"^[a-fA-F0-9]{40}$")
    hash_pattern_sha256 = re.compile(r"^[a-fA-F0-9]{64}$")

    if ip_pattern.match(ioc):
        ioc_type = "ip"
    elif domain_pattern.match(ioc):
        ioc_type = "domain"
    elif hash_pattern_md5.match(ioc) or hash_pattern_sha1.match(ioc) or hash_pattern_sha256.match(ioc):
        ioc_type = "file"
    else:
        ioc_type = "unknown"

    return {"ioc": ioc, "type": ioc_type}


In [35]:
# ====== Threat Analyzer Class ======
class ThreatAnalyzer:
    def __init__(self, config_file="threat_config.yaml"):
        with open(config_file, 'r') as file:
            config = yaml.safe_load(file)

        self.severity_mapping = config.get("severity_mapping", {})
        self.risk_weights = config.get("risk_weights", {})
        self.ioc_labels = config.get("ioc_labels", {})

    def generate_report_time(self):
        local_tz = pytz.timezone('Asia/Karachi')
        return datetime.now(local_tz).strftime("%Y-%m-%d %H:%M:%S")

    def analyze(self, vt_response):
        ioc = vt_response.get("ioc", "Unknown")
        ioc_type = vt_response.get("type", "Unknown").lower()
        vt_data = vt_response.get("vt_data", {})

        # Weighted score
        malicious = vt_data.get("malicious", 0)
        suspicious = vt_data.get("suspicious", 0)
        unrated = vt_data.get("unrated", 0)
        harmless = vt_data.get("harmless", 0)

        total = malicious + suspicious + unrated + harmless
        total = total if total else 1

        weighted_score = (
            malicious * self.risk_weights.get("malicious", 2) +
            suspicious * self.risk_weights.get("suspicious", 1) +
            unrated * self.risk_weights.get("unrated", 0.5) +
            harmless * self.risk_weights.get("harmless", 0)
        )

        risk_score = round((weighted_score / (total * 2)) * 100, 2)

        # Threat status
        if weighted_score >= self.risk_weights.get("malicious", 2) * 2:
            threat_status = "Malicious"
        elif weighted_score >= self.risk_weights.get("suspicious", 1) * 2:
            threat_status = "Suspicious"
        elif weighted_score >= self.risk_weights.get("unrated", 0.5) * 2:
            threat_status = "Unrated"
        else:
            threat_status = "Harmless"

        severity = self.severity_mapping.get(threat_status, "Unknown")

        label = self.ioc_labels.get(ioc_type, "Unknown IOC Type")

        # Generate detailed report
        report_time = self.generate_report_time()
        description = f"The IOC {ioc} ({label}) is associated with {ioc_type} activity."

        # Returning data in JSON format
        return {
            "ioc": ioc,
            "type": ioc_type,
            "threat_status": threat_status,
            "severity": severity,
            "risk_score": risk_score,
            "description": description,
            "report_generated_at": report_time
        }

In [36]:
from openai import OpenAI

client = OpenAI(
    api_key="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",
    base_url="https://api.groq.com/openai/v1"
)

In [37]:
def generateRemediation(ioc_data: dict):
  #Note for team:
  # Using json.dumps to pretty-print the IOC data before sending it to the model
  # This helps ensure the model interprets the input correctly, especially for nested structures.
  # Without pretty-printing, the input may be harder for the model to parse and could lead to inaccurate responses.
  ioc_str = json.dumps(ioc_data, indent=2)

  response = client.chat.completions.create(
      model="llama3-8b-8192",
      messages=[
          {
              "role": "system",
              "content": (
                  "You are a cybersecurity analyst. Your task is to verify if the provided IOC is malicious. "
                  "If it's benign (e.g., Google DNS, Microsoft domains), clearly state that no action is needed. "
                  "If it's malicious, provide only 4–6 concise and high-impact remediation steps in bullet points. "
                  "Avoid excessive explanation or repetition. Don't exceed 100 words total. Be direct."
              )
          },
          {
              "role": "user",
              "content": f"Here is the IOC data:\n\n{ioc_str}\n\nIs this IOC malicious? If so, list concise remediation steps."
          }
      ]
  )

  remediation = response.choices[0].message.content

  ioc_data["remediation"] = remediation

  return ioc_data

def analyzeThreatAndGenerateRemediation(vt_response):
  analyzer = ThreatAnalyzer(config_file="threat_config.yaml")

  result = analyzer.analyze(vt_response)

  result = generateRemediation(result)

  return result

In [38]:
def generate_pdf_report(result, filename="threat_report.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    width, height = letter

    c.setFont("Helvetica", 12)

    # Title
    c.drawString(100, height - 50, "Threat Analysis Report")
    c.drawString(100, height - 70, "-" * 23)

    # Initialize y_position for content
    y_position = height - 100
    margin_left = 100  # Left margin for content
    line_height = 14   # Line height for better readability

    # Helper function to wrap and draw text
    def draw_wrapped_text(y_position, text, line_height, indent=0):
        max_chars = 90
        words = text.split()
        current_line = ""
        for word in words:
            if len(current_line + word) < max_chars:
                current_line += word + " "
            else:
                c.drawString(margin_left + indent, y_position, current_line.strip())
                y_position -= line_height
                current_line = word + " "
                if y_position < 50:
                    c.showPage()
                    c.setFont("Helvetica", 12)
                    y_position = height - 50
        if current_line:
            c.drawString(margin_left + indent, y_position, current_line.strip())
            y_position -= line_height
        return y_position

    # Draw standard fields
    y_position = draw_wrapped_text(y_position, f"IOC: {result['ioc']}", line_height)
    y_position -= line_height 
    y_position = draw_wrapped_text(y_position, f"Type: {result['type']}", line_height)
    y_position -= line_height 
    y_position = draw_wrapped_text(y_position, f"Description: {result['description']}", line_height)
    y_position -= line_height 
    y_position = draw_wrapped_text(y_position, f"Threat Status: {result['threat_status']}", line_height)
    y_position -= line_height 
    y_position = draw_wrapped_text(
        y_position,
        f"Severity: {result['severity']} - This indicates a critical threat that requires immediate remediation to prevent potential harm to the organization's systems.",
        line_height
    )
    y_position -= line_height 
    y_position = draw_wrapped_text(
        y_position,
        f"Risk Score: {result['risk_score']}/100 - Calculated based on weighted malicious activity, suspicious behavior, and threat intelligence.",
        line_height
    )
    y_position -= line_height 
    # Draw Remediation steps cleanly
    y_position = draw_wrapped_text(y_position, "Remediation Steps:", line_height)
    y_position -= line_height 
    # Split remediation steps by bullet points
    steps = result['remediation'].split("•") if "•" in result['remediation'] else result['remediation'].split("*")
    for step in steps:
        step = step.strip()
        if step:
            y_position = draw_wrapped_text(y_position, f"• {step}", line_height, indent=10)
    y_position -= line_height 
    # Final line
    y_position = draw_wrapped_text(y_position, f"Report generated on: {result['report_generated_at']}", line_height)

    # Save the PDF
    c.save()
    print(f"PDF Report generated: {filename}")

In [39]:
def threat_analyzer_response(vt_response, response_type = "json"):
    result = analyzeThreatAndGenerateRemediation(vt_response)

    if response_type == "json":
      return result
    elif response_type == "pdf":
      generate_pdf_report(result, "threat_analysis_report.pdf")
    else:
      print("Invalid response type. Use 'json' or 'pdf'.")

In [40]:
from langgraph.graph import Graph
from typing import Any, Dict
from langgraph.graph import END

class Input_Handler:
    def __call__(self, state: Dict[str, Any]) -> Dict[str, Any]:
        try:
            print("Agent 1 received:", state)
            iocs = state.get("iocs", [])  # Get a list of IOCs

            # Initialize a list to hold the processed states for each IOC
            updated_states = []

            for ioc in iocs:
                check = checkioc(ioc)
                if check["type"] == "unknown":
                    raise ValueError(f"Invalid IOC format: {ioc}")

                # Update state with detected IOC and its type
                new_state = state.copy()  # Create a copy of the current state
                new_state.update(check)
                updated_states.append(new_state)

            # Return the list of updated states
            state["updated_states"] = updated_states
        except Exception as e:
            print(f"Error in Input_Handler: {e}")
            state["error"] = str(e)  # Add error information to state
        return state

    
class Threat_Intelligence_Fetcher:
    def __call__(self, state: Dict[str, Any]) -> Dict[str, Any]:
        try:
            print("Agent 2 received:", state)

            # Get the list of updated states with IOCs
            updated_states = state.get("updated_states", [])

            # Initialize a list to hold the results for each IOC
            results = []

            for s in updated_states:
                # Merge all data for each IOC
                vt_response = {
                    "ioc": s.get("ioc"),
                    "type": s.get("ioc_type"),
                    "vt_data": s.get("vt_data", {})
                }

                # Initialize the ThreatAnalyzer
                analyzer = ThreatAnalyzer(config_file="threat_config.yaml")

                # Analyze the data
                result = analyzer.analyze(vt_response)

                # Get the response type from state (default to 'json')
                response_type = s.get("response_type", "json")

                # Generate report based on response type
                if response_type == "pdf":
                    generate_pdf_report(result, f"{s.get('ioc')}_threat_analysis_report.pdf")
                else:
                    s["final_report"] = result  # Add the JSON report to state

                results.append(s)

            # Update the state with the processed results
            state["updated_states"] = results
        except Exception as e:
            print(f"Error in Threat_Intelligence_Fetcher: {e}")
            state["error"] = str(e)  # Add error information to state
        return state



class Report_Generator_Remediator:
    def __call__(self, state: Dict[str, Any]) -> Dict[str, Any]:
        try:
            print("Agent 3 received:", state)

            # Get the list of updated states with reports
            updated_states = state.get("updated_states", [])

            for s in updated_states:
                # Merge all data for each IOC
                vt_response = {
                    "ioc": s.get("ioc"),
                    "type": s.get("ioc_type"),
                    "vt_data": s.get("vt_data", {})
                }

                # Generate final threat report for each IOC
                response_type = s.get("response_type", "json")  # user can define if they want pdf
                result = threat_analyzer_response(vt_response, response_type=response_type)

                s["final_report"] = result  # Add final report in state

            # Return the updated state with the final reports for each IOC
            state["updated_states"] = updated_states
        except Exception as e:
            print(f"Error in Report_Generator_Remediator: {e}")
            state["error"] = str(e)  # Add error information to state
        return state

agent1 = Input_Handler()
agent2 = Threat_Intelligence_Fetcher()
agent3 = Report_Generator_Remediator()

graph = Graph()

graph.add_node("agent1", agent1)
graph.add_node("agent2", agent2)
graph.add_node("agent3", agent3)

graph.set_entry_point("agent1")
graph.add_edge("agent1", "agent2")
graph.add_edge("agent2", "agent3")
graph.add_edge("agent3", END)

compiled_graph = graph.compile()


In [41]:
# Step 1: Sample VT response for a malicious domain
sample_vt_response = {
    "ioc": "maliciousdomain.xyz",
    "type": "domain",
    "vt_data": {
        "malicious": 28,
        "suspicious": 3,
        "harmless": 2,
        "unrated": 0,
        "last_analysis_date": "2025-04-04 12:00:00"
    }
}
result = analyzeThreatAndGenerateRemediation(sample_vt_response)

# Step 3: Print JSON result (optional)
import json
print("Full Analysis Result:\n")
print(json.dumps(result, indent=2))

pdf_filename = f"{result['ioc']}_threat_analysis_report.pdf"
generate_pdf_report(result, pdf_filename)

print(f"\n✅ PDF Report generated: {pdf_filename}")

Full Analysis Result:

{
  "ioc": "maliciousdomain.xyz",
  "type": "domain",
  "threat_status": "Malicious",
  "severity": "High",
  "risk_score": 89.39,
  "description": "The IOC maliciousdomain.xyz (Domain) is associated with domain activity.",
  "report_generated_at": "2025-04-30 21:32:15",
  "remediation": "This IOC is malicious. The domain \"maliciousdomain.xyz\" is flagged as high-risk and is associated with malicious activity.\n\n**Remediation Steps:**\n\n\u2022 Block access to the domain \"maliciousdomain.xyz\" at the network level.\n\u2022 Update DNS records to prevent DNS queries to this domain.\n\u2022 Run a full system scan with your antivirus software.\n\u2022 Verify for any suspicious or unauthorized software installations.\n\u2022 Monitor system logs for any signs of compromise."
}
PDF Report generated: maliciousdomain.xyz_threat_analysis_report.pdf

✅ PDF Report generated: maliciousdomain.xyz_threat_analysis_report.pdf
